# Curanex: TB chest X-ray (GPU)

An earlier frozen-feature TB layer failed across sources (AUC 0.51-0.63). **Rules fixed before running (2026-11-11):**
- Train on TBX11K (China): TB against healthy and sick-but-not-TB; 15% stratified validation; whole network fine-tuned;
  threshold = highest with validation sensitivity >= 0.85.
- **Outside test, scored once:** NLM Shenzhen (China) and Montgomery County (USA) sets, TB against normal (file name ends _1 = TB).
- Verified if outside sensitivity and AUC are both >= 0.80; live at >= 0.75; otherwise off.

In [ ]:
import glob, hashlib, json, os, random
import numpy as np, torch, torch.nn as nn
from PIL import Image
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from torchvision import transforms as T
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

torch.manual_seed(42); random.seed(42); np.random.seed(42)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEV, torch.cuda.get_device_name(0) if DEV == "cuda" else "")
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]


def dhash(p):
    a = np.asarray(Image.open(p).convert("L").resize((9, 8)), dtype=np.float32)
    return int("".join("1" if b else "0" for b in (a[:, 1:] > a[:, :-1]).ravel()), 2)


def near(h, bank):
    return bool((np.unpackbits(np.bitwise_xor(bank, np.uint64(h)).view(np.uint8).reshape(-1, 8), axis=1).sum(1) <= 6).any())


train_tf = T.Compose([T.RandomResizedCrop(224, scale=(0.6, 1.0)), T.RandomRotation(20), T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
                      T.ColorJitter(brightness=0.35, contrast=0.35, saturation=0.3, hue=0.05), T.RandomApply([T.GaussianBlur(5)], p=0.3),
                      T.ToTensor(), T.Normalize(MEAN, STD)])
eval_tf = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor(), T.Normalize(MEAN, STD)])  # what the app does


class DS(torch.utils.data.Dataset):
    def __init__(self, items, tf):
        self.items, self.tf = items, tf

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        return self.tf(Image.open(self.items[i][0]).convert("RGB")), self.items[i][1]


def loader(items, tf, shuffle=False):
    return torch.utils.data.DataLoader(DS(items, tf), batch_size=32, shuffle=shuffle, num_workers=2)


def train_model(rows, n_classes, epochs):
    y = np.array([r[1] for r in rows])
    tr, va = train_test_split(np.arange(len(rows)), test_size=0.15, random_state=42, stratify=y)
    net = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
    net.classifier[1] = nn.Linear(net.classifier[1].in_features, n_classes)
    net = net.to(DEV)
    counts = np.bincount(y[tr], minlength=n_classes)
    loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(len(tr) / (n_classes * np.maximum(counts, 1)), dtype=torch.float32, device=DEV))
    opt = torch.optim.AdamW(net.parameters(), lr=3e-4, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    dl_tr, dl_va = loader([rows[i] for i in tr], train_tf, True), loader([rows[i] for i in va], eval_tf)

    def predict(dl):
        net.eval()
        out = []
        with torch.no_grad():
            for x, _ in dl:
                out.append(torch.softmax(net(x.to(DEV)), 1).cpu().numpy())
        return np.concatenate(out)

    best, state = -1, None
    for ep in range(epochs):
        net.train()
        for x, lab in dl_tr:
            opt.zero_grad()
            loss = loss_fn(net(x.to(DEV)), lab.to(DEV))
            loss.backward()
            opt.step()
        sched.step()
        pv = predict(dl_va)
        auc = roc_auc_score(y[va], pv[:, 1] if n_classes == 2 else pv, **({} if n_classes == 2 else {"multi_class": "ovr", "average": "macro"}))
        print(f"epoch {ep + 1}: validation AUC {auc:.4f}")
        if auc > best:
            best, state = auc, {k: v.detach().cpu().clone() for k, v in net.state_dict().items()}
    net.load_state_dict(state)
    pv = predict(dl_va)
    thresholds = []
    for i in range(n_classes):
        pos = pv[y[va] == i, i]
        thresholds.append(float(max(v for v in np.unique(pos) if (pos >= v).mean() >= 0.85)) if len(pos) else 0.5)
    return net, predict, thresholds, best


def report(y, s, t):
    f = s >= t
    r = {"n": int(len(y)), "positives": int(y.sum()), "sensitivity": round(float(f[y == 1].mean()), 3),
         "specificity": round(float((~f[y == 0]).mean()), 3), "auc": round(float(roc_auc_score(y, s)), 3)}
    low = min(r["sensitivity"], r["auc"])
    r["status"] = "verified" if low >= 0.80 else "live" if low >= 0.75 else "off"
    return r


def export(net, name, meta):
    class Full(nn.Module):
        def __init__(self, n):
            super().__init__()
            self.n = n

        def forward(self, x):
            return torch.softmax(self.n(x), 1)

    full = Full(net.cpu()).eval()
    path = f"/kaggle/working/{name}.onnx"
    args = dict(input_names=["image"], output_names=["probs"], opset_version=17, dynamic_axes={"image": {0: "n"}})
    try:
        torch.onnx.export(full, torch.randn(1, 3, 224, 224), path, dynamo=False, **args)
    except TypeError:
        torch.onnx.export(full, torch.randn(1, 3, 224, 224), path, **args)
    meta |= {"mean": MEAN, "std": STD, "onnx_sha256": hashlib.sha256(open(path, "rb").read()).hexdigest()}
    json.dump(meta, open(f"/kaggle/working/{name}.json", "w"), indent=1)
    print("wrote", name)

In [ ]:
import pandas as pd
LABELS = ["TB", "No_TB"]
d = pd.read_csv(glob.glob("/kaggle/input/**/tbx11k-simplified/data.csv", recursive=True)[0]).drop_duplicates("fname")
img_dir = os.path.join(os.path.dirname(glob.glob("/kaggle/input/**/tbx11k-simplified/data.csv", recursive=True)[0]), "images")
rows = [(os.path.join(img_dir, r.fname), 0 if r.target == "tb" else 1) for r in d.itertuples() if os.path.exists(os.path.join(img_dir, r.fname))]
print("TBX11K training images:", len(rows), "TB:", sum(1 for _, y in rows if y == 0))
test = []
for f in glob.glob("/kaggle/input/**/images/images/*.png", recursive=True):
    name = os.path.basename(f)
    if name.startswith(("CHNCXR", "MCUCXR")):
        test.append((f, 1 if name.endswith("_1.png") else 0))
print("NLM test images:", len(test), "TB:", sum(y for _, y in test))

In [ ]:
net, predict, thr, best = train_model(rows, 2, epochs=10)
pt = predict(loader(test, eval_tf))[:, 0]
yt = np.array([y for _, y in test])
res = report(yt, pt, thr[0])
print("OUTSIDE TEST on NLM Shenzhen + Montgomery (once):", res)
export(net, "tb_xray", {"id": "tb_xray", "title": "TB chest X-ray", "labels": LABELS, "thresholds": thr, "validation_auc": round(best, 4),
                        "outside_test": res, "images": len(rows),
                        "trained_on": f"{len(rows)} TBX11K chest X-rays (China), full fine-tuning on a Kaggle GPU",
                        "outside_set": "NLM Shenzhen (China) and Montgomery County (USA), TB vs normal"})